In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load CSVs
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')

# Ensure types
df_metric['timestamp'] = df_metric['timestamp'].astype(int)
df_trace['timestamp'] = df_trace['timestamp'].astype(int)

# Overall min/max timestamps per file (converted to UTC+8)
def to_local_str(ts):
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

metrics_overall_min = df_metric['timestamp'].min() if not df_metric.empty else None
metrics_overall_max = df_metric['timestamp'].max() if not df_metric.empty else None
trace_overall_min = df_trace['timestamp'].min() if not df_trace.empty else None
trace_overall_max = df_trace['timestamp'].max() if not df_trace.empty else None

metrics_range_str = (f"metric.csv overall_min={to_local_str(metrics_overall_min)}"
                     f", overall_max={to_local_str(metrics_overall_max)}") if metrics_overall_min is not None else "metric.csv is empty"
trace_range_str = (f"trace.csv overall_min={to_local_str(trace_overall_min)}"
                   f", overall_max={to_local_str(trace_overall_max)}") if trace_overall_min is not None else "trace.csv is empty"

# Aggregate metric.csv by (cmdb_id, kpi_name)
if not df_metric.empty:
    g = df_metric.groupby(['cmdb_id', 'kpi_name'])
    metrics_agg = (
        g.agg(
            count=('value', 'size'),
            min_timestamp=('timestamp', 'min'),
            max_timestamp=('timestamp', 'max'),
            p95_value=('value', lambda x: float(np.percentile(x.dropna(), 95)) if x.dropna().size>0 else np.nan)
        )
        .reset_index()
    )
    # convert timestamps to UTC+8 strings
    metrics_agg['min_timestamp'] = metrics_agg['min_timestamp'].apply(to_local_str)
    metrics_agg['max_timestamp'] = metrics_agg['max_timestamp'].apply(to_local_str)
    metrics_agg = metrics_agg.sort_values('count', ascending=False).head(50)
else:
    metrics_agg = pd.DataFrame(columns=['cmdb_id','kpi_name','count','min_timestamp','max_timestamp','p95_value'])

# Aggregate trace.csv by (cmdb_id, trace_name)
if not df_trace.empty:
    gt = df_trace.groupby(['cmdb_id', 'trace_name'])
    trace_agg = (
        gt.agg(
            count=('value', 'size'),
            min_timestamp=('timestamp', 'min'),
            max_timestamp=('timestamp', 'max'),
            p95_value=('value', lambda x: float(np.percentile(x.dropna(), 95)) if x.dropna().size>0 else np.nan)
        )
        .reset_index()
    )
    trace_agg['min_timestamp'] = trace_agg['min_timestamp'].apply(to_local_str)
    trace_agg['max_timestamp'] = trace_agg['max_timestamp'].apply(to_local_str)
    trace_agg = trace_agg.sort_values('count', ascending=False).head(50)
else:
    trace_agg = pd.DataFrame(columns=['cmdb_id','trace_name','count','min_timestamp','max_timestamp','p95_value'])

# Prepare compact display: limit columns and rows (already limited to 50)
metrics_agg_display = metrics_agg[['cmdb_id','kpi_name','count','min_timestamp','max_timestamp','p95_value']]
trace_agg_display = trace_agg[['cmdb_id','trace_name','count','min_timestamp','max_timestamp','p95_value']]

# Display results (use variable names, separated by commas)
metrics_agg_display, trace_agg_display, metrics_range_str, trace_range_str

```
Out[1]:
```
summary = (
    "Summary of telemetry aggregation results:\n\n"
    "metric.csv\n"
    "- Overall time range (UTC+8): 2020-05-26 02:30:00 to 2020-05-26 03:30:00\n"
    "- Top aggregates by sample count (examples):\n"
    "  * os_009 / Disk_io_util — count=61, p95=0.0\n"
    "  * os_017 / Disk_io_util — count=61, p95=0.0\n"
    "  * osb_001 / app.avg_time — count=61, p95≈0.00949\n"
    "  * osb_001 / app.num — count=61, p95=506.0\n"
    "  * osb_001 / app.succee_num — count=61, p95=506.0\n"
    "- Top sample count seen: 61 (many cmdb_id/kpi_name groups)\n\n"
    "trace.csv\n"
    "- Overall time range (UTC+8): 2020-05-26 03:00:00 to 2020-05-26 03:29:00\n"
    "- Top aggregates by sample count (examples):\n"
    "  * docker_001 / trace.from_os_022.remoteprocess.duration_mean — count=30, p95≈0.5817\n"
    "  * docker_001 / trace.from_os_022.remoteprocess.duration_p95 — count=30, p95≈1.8503\n"
    "  * docker_001 / trace.from_os_022.remoteprocess.row_count — count=30, p95=112.1\n"
    "  * docker_001 / trace.self.csf.duration_mean — count=30, p95≈0.05616\n"
    "  * docker_002 / trace.self.local.row_count — count=30, p95=1556.8\n"
    "- Top sample count seen: 30 (many cmdb_id/trace_name groups)\n\n"
    "Notes:\n"
    "- Results are compact examples from the top 50 aggregated groups per file.\n"
    "- p95 values shown are the global 95th percentiles computed over each full series in the file.\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id                  kpi_name  count            min_timestamp            max_timestamp   p95_value
1077   os_009              Disk_io_util     61  2020-05-26 02:30:00 CST  2020-05-26 03:30:00 CST    0.000000
1485   os_017              Disk_io_util     61  2020-05-26 02:30:00 CST  2020-05-26 03:30:00 CST    0.000000
1781  osb_001              app.avg_time     61  2020-05-26 02:30:00 CST  2020-05-26 03:30:00 CST    0.009488
1782  osb_001                   app.num     61  2020-05-26 02:30:00 CST  2020-05-26 03:30:00 CST  506.000000
1783  osb_001            app.succee_num     61  2020-05-26 02:30:00 CST  2020-05-26 03:30:00 CST  506.000000
...       ...                       ...    ...                      ...                      ...         ...
783    os_003  Incoming_network_traffic     60  2020-05-26 02:30:00 CST  2020-05-26 03:29:00 CST    2.417923
1586   os_019                Disk_await     60  2020-05-26 02:30:00 CST  2020-05-26 03:29:00 CST    1.000000
1585   os_019             Disk_avgqu_sz     60  2020-05-26 02:30:00 CST  2020-05-26 03:29:00 CST    0.034534
1538   os_018               Disk_rd_kbs     60  2020-05-26 02:30:00 CST  2020-05-26 03:29:00 CST    0.000000
1608   os_019  Outgoing_network_traffic     60  2020-05-26 02:30:00 CST  2020-05-26 03:29:00 CST    0.055078

[50 rows x 6 columns],        cmdb_id                                       trace_name  count            min_timestamp            max_timestamp    p95_value
0   docker_001    trace.from_os_022.remoteprocess.duration_mean     30  2020-05-26 03:00:00 CST  2020-05-26 03:29:00 CST     0.581684
1   docker_001     trace.from_os_022.remoteprocess.duration_p95     30  2020-05-26 03:00:00 CST  2020-05-26 03:29:00 CST     1.850295
2   docker_001       trace.from_os_022.remoteprocess.error_rate     30  2020-05-26 03:00:00 CST  2020-05-26 03:29:00 CST     0.000000
3   docker_001        trace.from_os_022.remoteprocess.row_count     30  2020-05-26 03:00:00 CST  2020-05-26 03:29:00 CST   112.100000
4   docker_001                     trace.self.csf.duration_mean     30  2020-05-26 03:00:00 CST  2020-05-26 03:29:00 CST     0.056159
..         ...                                              ...    ...                      ...                      ...          ...
45  docker_002                    trace.self.local.duration_p95     30  2020-05-26 03:00:00 CST  2020-05-26 03:29:00 CST     0.096550
46  docker_002                      trace.self.local.error_rate     30  2020-05-26 03:00:00 CST  2020-05-26 03:29:00 CST     0.000000
47  docker_002                       trace.self.local.row_count     30  2020-05-26 03:00:00 CST  2020-05-26 03:29:00 CST  1556.800000
48  docker_002  trace.to_docker_007.remoteprocess.duration_mean     30  2020-05-26 03:00:00 CST  2020-05-26 03:29:00 CST     0.030181
49  docker_002   trace.to_docker_007.remoteprocess.duration_p95     30  2020-05-26 03:00:00 CST  2020-05-26 03:29:00 CST     0.054252

[50 rows x 6 columns], 'metric.csv overall_min=2020-05-26 02:30:00 CST, overall_max=2020-05-26 03:30:00 CST', 'trace.csv overall_min=2020-05-26 03:00:00 CST, overall_max=2020-05-26 03:29:00 CST')

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

The Executor failed to complete the instruction, please re-write a new instruction for Executor.

```
Out[2]:
```
The Executor failed to complete the instruction, please re-write a new instruction for Executor.```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse df_metric if available in kernel, otherwise load
if 'df_metric' not in globals():
    df_metric = pd.read_csv('metric.csv')
# Ensure correct types
df_metric['timestamp'] = df_metric['timestamp'].astype(int)
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# 1) Compute global P95 and P5 per (cmdb_id, kpi_name) over the full series
thresholds = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(
        p95_value=lambda x: float(np.percentile(x.dropna(), 95)) if x.dropna().size>0 else np.nan,
        p5_value = lambda x: float(np.percentile(x.dropna(), 5))  if x.dropna().size>0 else np.nan,
    )
    .reset_index()
)

# 2) Filter raw metric.csv to incident window (UTC+8) 2020-05-26 03:00:00 .. 2020-05-26 03:30:00 inclusive
start_local = pd.Timestamp('2020-05-26 03:00:00').tz_localize(tz)
end_local   = pd.Timestamp('2020-05-26 03:30:00').tz_localize(tz)
# convert to UTC epoch seconds for comparison with timestamps stored as epoch seconds
start_ts = int(start_local.tz_convert('UTC').timestamp())
end_ts   = int(end_local.tz_convert('UTC').timestamp())

df_window = df_metric[(df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)].copy()

# 3) Merge thresholds and mark anomalies
df_w = df_window.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# mark anomaly_type
df_w['anomaly_type'] = np.where(df_w['value'] >= df_w['p95_value'], 'high',
                               np.where(df_w['value'] <= df_w['p5_value'], 'low', None))

df_anom = df_w[df_w['anomaly_type'].notna()].copy()

# 4) Aggregate anomalies by (cmdb_id, kpi_name, anomaly_type)
if not df_anom.empty:
    agg = (
        df_anom
        .groupby(['cmdb_id', 'kpi_name', 'anomaly_type'])
        .agg(
            anomaly_count=('value','size'),
            earliest_anomaly_timestamp=('timestamp','min'),
            max_value=('value','max'),
            min_value=('value','min'),
            p95_value=('p95_value','first'),
            p5_value=('p5_value','first'),
        )
        .reset_index()
    )
    # convert earliest timestamp to UTC+8 readable string
    agg['earliest_anomaly_timestamp'] = agg['earliest_anomaly_timestamp'].apply(
        lambda ts: pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    )
    # compute severity_score safely
    def compute_severity(row):
        if row['anomaly_type'] == 'high':
            denom = row['p95_value']
            if pd.isna(denom) or denom == 0:
                return np.nan
            return float(row['max_value']) / float(denom)
        else:  # low
            denom = row['min_value']
            if pd.isna(denom) or denom == 0:
                return np.nan
            return float(row['p5_value']) / float(denom)

    agg['severity_score'] = agg.apply(compute_severity, axis=1)

    # reorder columns per request
    agg_display = agg[[
        'cmdb_id','kpi_name','anomaly_type','anomaly_count','earliest_anomaly_timestamp',
        'max_value','min_value','p95_value','p5_value','severity_score'
    ]]

    # 5) Sort result by anomaly_count desc, then severity_score desc and return top 20
    agg_top = agg_display.sort_values(['anomaly_count','severity_score'], ascending=[False, False]).head(20).reset_index(drop=True)

    # 6) summary numbers
    total_groups = int(agg.shape[0])
    total_points = int(agg['anomaly_count'].sum())
else:
    agg_top = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','anomaly_type','anomaly_count','earliest_anomaly_timestamp',
        'max_value','min_value','p95_value','p5_value','severity_score'
    ])
    total_groups = 0
    total_points = 0

# Display compact results: top anomalies table, and two summary numbers
agg_top, total_groups, total_points

```
Out[3]:
```
summary = (
    "Summary of metric anomaly detection (incident window 2020-05-26 03:00:00 to 2020-05-26 03:30:00 UTC+8):\n"
    "- Total distinct anomaly groups detected: 2804\n"
    "- Total anomaly points (rows): 24652\n\n"
    "Top example anomaly groups (from aggregated results):\n"
    "1) os_009 / Disk_io_util — anomaly_type=high, anomaly_count=31, p95=0.0 (severity undefined due to p95=0)\n"
    "2) os_010 / Disk_io_util — anomaly_type=high, anomaly_count=31, p95=0.0 (severity undefined due to p95=0)\n"
    "3) os_017 / Disk_io_util — anomaly_type=high, anomaly_count=31, p95=0.0 (severity undefined due to p95=0)\n"
    "4) os_002 / Memory_used_pct — anomaly_type=high, anomaly_count=30, p95=77.29, max_value=77.30, severity≈1.00013\n"
    "5) db_001 / Hang and db_001 / On_Off_State — anomaly_type=high, anomaly_count=30, p95=1.0, severity=1.0\n\n"
    "Notes:\n"
    "- P95 and P5 were computed over the full series before filtering to the incident window.\n"
    "- Many Disk_io_util groups have p95=0 causing severity to be undefined (division-by-zero). Several db_* and MEM-related KPIs have p95 equal to observed values, yielding severity≈1.0.\n"
    "- The table returned was the top aggregated anomaly groups (sorted by anomaly_count then severity)."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id         kpi_name anomaly_type  anomaly_count earliest_anomaly_timestamp   max_value   min_value   p95_value    p5_value  severity_score
0   os_009     Disk_io_util         high             31    2020-05-26 03:00:00 CST    0.000000    0.000000    0.000000    0.000000             NaN
1   os_010     Disk_io_util         high             31    2020-05-26 03:00:00 CST    0.000000    0.000000    0.000000    0.000000             NaN
2   os_017     Disk_io_util         high             31    2020-05-26 03:00:00 CST    0.000000    0.000000    0.000000    0.000000             NaN
3   os_002  Memory_used_pct         high             30    2020-05-26 03:00:00 CST   77.300000   77.290000   77.290000   77.290000        1.000129
4   db_001             Hang         high             30    2020-05-26 03:00:00 CST    1.000000    1.000000    1.000000    1.000000        1.000000
5   db_001     On_Off_State         high             30    2020-05-26 03:00:00 CST    1.000000    1.000000    1.000000    1.000000        1.000000
6   db_002             Hang         high             30    2020-05-26 03:00:00 CST    1.000000    1.000000    1.000000    1.000000        1.000000
7   db_002        MEM_Total         high             30    2020-05-26 03:00:00 CST  381.902264  381.902264  381.902264  381.902264        1.000000
8   db_002     On_Off_State         high             30    2020-05-26 03:00:00 CST    1.000000    1.000000    1.000000    1.000000        1.000000
9   db_003             Hang         high             30    2020-05-26 03:00:00 CST    1.000000    1.000000    1.000000    1.000000        1.000000
10  db_003     On_Off_State         high             30    2020-05-26 03:00:00 CST    1.000000    1.000000    1.000000    1.000000        1.000000
11  db_004             Hang         high             30    2020-05-26 03:00:00 CST    1.000000    1.000000    1.000000    1.000000        1.000000
12  db_004        MEM_Total         high             30    2020-05-26 03:00:00 CST  381.902264  381.902264  381.902264  381.902264        1.000000
13  db_004     On_Off_State         high             30    2020-05-26 03:00:00 CST    1.000000    1.000000    1.000000    1.000000        1.000000
14  db_005             Hang         high             30    2020-05-26 03:00:00 CST    1.000000    1.000000    1.000000    1.000000        1.000000
15  db_005        MEM_Total         high             30    2020-05-26 03:00:00 CST  506.929929  506.929929  506.929929  506.929929        1.000000
16  db_005     On_Off_State         high             30    2020-05-26 03:00:00 CST    1.000000    1.000000    1.000000    1.000000        1.000000
17  db_006             Hang         high             30    2020-05-26 03:00:00 CST    1.000000    1.000000    1.000000    1.000000        1.000000
18  db_006        MEM_Total         high             30    2020-05-26 03:00:00 CST  508.252410  508.252410  508.252410  508.252410        1.000000
19  db_006     On_Off_State         high             30    2020-05-26 03:00:00 CST    1.000000    1.000000    1.000000    1.000000        1.000000, 2804, 24652)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# reuse tz and df_trace if available
if 'tz' not in globals():
    tz = pytz.timezone('Asia/Shanghai')
if 'df_trace' not in globals():
    df_trace = pd.read_csv('trace.csv')

# ensure types
df_trace['timestamp'] = df_trace['timestamp'].astype(int)
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')

# 1) compute global P95 and P5 per (cmdb_id, trace_name) over full series
trace_thresholds = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'])['value']
    .agg(
        p95_value=lambda x: float(np.percentile(x.dropna(), 95)) if x.dropna().size>0 else np.nan,
        p5_value = lambda x: float(np.percentile(x.dropna(), 5))  if x.dropna().size>0 else np.nan,
    )
    .reset_index()
)

# 2) filter raw trace rows to incident window (UTC+8) 2020-05-26 03:00:00 .. 2020-05-26 03:30:00 inclusive
start_local = pd.Timestamp('2020-05-26 03:00:00').tz_localize(tz)
end_local   = pd.Timestamp('2020-05-26 03:30:00').tz_localize(tz)
start_ts = int(start_local.tz_convert('UTC').timestamp())
end_ts   = int(end_local.tz_convert('UTC').timestamp())

df_trace_window = df_trace[(df_trace['timestamp'] >= start_ts) & (df_trace['timestamp'] <= end_ts)].copy()

# 3) merge thresholds and mark anomalies
df_t = df_trace_window.merge(trace_thresholds, on=['cmdb_id', 'trace_name'], how='left')

df_t['anomaly_type'] = np.where(df_t['value'] >= df_t['p95_value'], 'high',
                               np.where(df_t['value'] <= df_t['p5_value'], 'low', None))

df_trace_anom = df_t[df_t['anomaly_type'].notna()].copy()

# 4) aggregate anomalies by (cmdb_id, trace_name, anomaly_type)
if not df_trace_anom.empty:
    agg_t = (
        df_trace_anom
        .groupby(['cmdb_id', 'trace_name', 'anomaly_type'])
        .agg(
            anomaly_count=('value','size'),
            earliest_anomaly_timestamp=('timestamp','min'),
            max_value=('value','max'),
            min_value=('value','min'),
            p95_value=('p95_value','first'),
            p5_value=('p5_value','first'),
        )
        .reset_index()
    )

    # convert earliest timestamp to UTC+8 string
    agg_t['earliest_anomaly_timestamp'] = agg_t['earliest_anomaly_timestamp'].apply(
        lambda ts: pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    )

    # compute severity_score safely
    def compute_severity_trace(row):
        if row['anomaly_type'] == 'high':
            denom = row['p95_value']
            if pd.isna(denom) or denom == 0:
                return np.nan
            return float(row['max_value']) / float(denom)
        else:  # low
            denom = row['min_value']
            if pd.isna(denom) or denom == 0:
                return np.nan
            return float(row['p5_value']) / float(denom)

    agg_t['severity_score'] = agg_t.apply(compute_severity_trace, axis=1)

    agg_t_display = agg_t[[
        'cmdb_id','trace_name','anomaly_type','anomaly_count','earliest_anomaly_timestamp',
        'max_value','min_value','p95_value','p5_value','severity_score'
    ]]

    # 5) sort and keep top 20
    trace_agg_top = agg_t_display.sort_values(['anomaly_count','severity_score'], ascending=[False, False]).head(20).reset_index(drop=True)

    # 6) summary numbers
    total_groups_trace = int(agg_t.shape[0])
    total_points_trace = int(agg_t['anomaly_count'].sum())
else:
    trace_agg_top = pd.DataFrame(columns=[
        'cmdb_id','trace_name','anomaly_type','anomaly_count','earliest_anomaly_timestamp',
        'max_value','min_value','p95_value','p5_value','severity_score'
    ])
    total_groups_trace = 0
    total_points_trace = 0

# Display compact results: aggregated top anomalies, and two summary numbers
trace_agg_top, total_groups_trace, total_points_trace

```
Out[4]:
```
Summary of trace anomaly detection (incident window 2020-05-26 03:00:00 — 2020-05-26 03:30:00 UTC+8):

- Totals:
  - Distinct anomaly groups detected: 319
  - Total anomaly points (rows): 2,117

- Key observations:
  - Many top anomaly groups are per-docker trace metrics with anomaly_count = 30 (the trace data covers 03:00–03:29 in the file, so 30 samples per group).
  - Example top groups:
    - docker_005 / trace.self.jdbc.duration_p95 — high, anomaly_count=30, p95=0.003, max=0.003, severity=1.0
    - docker_007 / trace.self.jdbc.duration_p95 — high, anomaly_count=30, p95=0.003, severity=1.0
    - docker_008 / trace.self.jdbc.duration_p95 — high, anomaly_count=30, p95=0.003, severity=1.0
    - Many docker_X / *.error_rate groups (e.g., docker_001 trace.from_os_022.remoteprocess.error_rate) are flagged high with count=30 but have p95=0 and max=0, so severity is undefined (division-by-zero).
  - P95 and P5 were computed globally over each full (cmdb_id, trace_name) series before filtering to the incident window.

- Notes:
  - Several trace metrics have zero-valued thresholds (p95 or p5 = 0), causing severity_score to be NaN when dividing by zero; these should be interpreted with caution.
  - Results returned are aggregated anomaly groups (top 20 shown in the run), not raw time series.

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                    trace_name anomaly_type  anomaly_count earliest_anomaly_timestamp  max_value  min_value  p95_value  p5_value  severity_score
0   docker_005                  trace.self.jdbc.duration_p95         high             30    2020-05-26 03:00:00 CST      0.003      0.003      0.003     0.003             1.0
1   docker_007                  trace.self.jdbc.duration_p95         high             30    2020-05-26 03:00:00 CST      0.003      0.003      0.003     0.003             1.0
2   docker_008                  trace.self.jdbc.duration_p95         high             30    2020-05-26 03:00:00 CST      0.003      0.003      0.003     0.003             1.0
3   docker_001    trace.from_os_022.remoteprocess.error_rate         high             30    2020-05-26 03:00:00 CST      0.000      0.000      0.000     0.000             NaN
4   docker_001                     trace.self.csf.error_rate         high             30    2020-05-26 03:00:00 CST      0.000      0.000      0.000     0.000             NaN
5   docker_001               trace.self.flyremote.error_rate         high             30    2020-05-26 03:00:00 CST      0.000      0.000      0.000     0.000             NaN
6   docker_001                    trace.self.jdbc.error_rate         high             30    2020-05-26 03:00:00 CST      0.000      0.000      0.000     0.000             NaN
7   docker_001                   trace.self.local.error_rate         high             30    2020-05-26 03:00:00 CST      0.000      0.000      0.000     0.000             NaN
8   docker_001  trace.to_docker_007.remoteprocess.error_rate         high             30    2020-05-26 03:00:00 CST      0.000      0.000      0.000     0.000             NaN
9   docker_001  trace.to_docker_008.remoteprocess.error_rate         high             30    2020-05-26 03:00:00 CST      0.000      0.000      0.000     0.000             NaN
10  docker_002    trace.from_os_022.remoteprocess.error_rate         high             30    2020-05-26 03:00:00 CST      0.000      0.000      0.000     0.000             NaN
11  docker_002                     trace.self.csf.error_rate         high             30    2020-05-26 03:00:00 CST      0.000      0.000      0.000     0.000             NaN
12  docker_002               trace.self.flyremote.error_rate         high             30    2020-05-26 03:00:00 CST      0.000      0.000      0.000     0.000             NaN
13  docker_002                    trace.self.jdbc.error_rate         high             30    2020-05-26 03:00:00 CST      0.000      0.000      0.000     0.000             NaN
14  docker_002                   trace.self.local.error_rate         high             30    2020-05-26 03:00:00 CST      0.000      0.000      0.000     0.000             NaN
15  docker_002  trace.to_docker_007.remoteprocess.error_rate         high             30    2020-05-26 03:00:00 CST      0.000      0.000      0.000     0.000             NaN
16  docker_002  trace.to_docker_008.remoteprocess.error_rate         high             30    2020-05-26 03:00:00 CST      0.000      0.000      0.000     0.000             NaN
17  docker_003    trace.from_os_021.remoteprocess.error_rate         high             30    2020-05-26 03:00:00 CST      0.000      0.000      0.000     0.000             NaN
18  docker_003                     trace.self.csf.error_rate         high             30    2020-05-26 03:00:00 CST      0.000      0.000      0.000     0.000             NaN
19  docker_003               trace.self.flyremote.error_rate         high             30    2020-05-26 03:00:00 CST      0.000      0.000      0.000     0.000             NaN, 319, 2117)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# Setup timezone
tz = pytz.timezone('Asia/Shanghai')

# Load trace.csv (reuse if present)
if 'df_trace' not in globals():
    df_trace = pd.read_csv('trace.csv')

# Ensure correct types
df_trace['timestamp'] = df_trace['timestamp'].astype(int)
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')

# 1) Compute global P95 and P5 per (cmdb_id, trace_name) over full series (explicitly compute to ensure column names)
trace_thresholds = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'])['value']
    .agg(
        global_p95_value=lambda x: float(np.percentile(x.dropna(), 95)) if x.dropna().size > 0 else np.nan,
        global_p5_value = lambda x: float(np.percentile(x.dropna(), 5))  if x.dropna().size > 0 else np.nan,
    )
    .reset_index()
)

# 2) Filter to incident window (UTC+8) 2020-05-26 03:00:00 .. 2020-05-26 03:30:00 inclusive
start_local = pd.Timestamp('2020-05-26 03:00:00').tz_localize(tz)
end_local   = pd.Timestamp('2020-05-26 03:30:00').tz_localize(tz)
start_ts = int(start_local.tz_convert('UTC').timestamp())
end_ts   = int(end_local.tz_convert('UTC').timestamp())

df_window = df_trace[(df_trace['timestamp'] >= start_ts) & (df_trace['timestamp'] <= end_ts)].copy()

# 3) Select trace_name values containing substrings (case-insensitive)
substrs = ['jdbc', 'remoteprocess', 'to_db', 'db']
pattern = '|'.join([s.lower() for s in substrs])
mask = df_window['trace_name'].str.lower().str.contains(pattern, na=False)
df_sel = df_window[mask].copy()

# Merge global thresholds
df_sel = df_sel.merge(trace_thresholds, on=['cmdb_id','trace_name'], how='left')

# 4) Aggregate per (cmdb_id, trace_name)
if not df_sel.empty:
    agg = (
        df_sel
        .groupby(['cmdb_id','trace_name'])
        .agg(
            sample_count_in_window=('value','size'),
            earliest_timestamp_in_window=('timestamp','min'),
            mean_value_in_window=('value','mean'),
            max_value_in_window=('value','max'),
            global_p95_value=('global_p95_value','first'),
            global_p5_value =('global_p5_value','first'),
        )
        .reset_index()
    )

    # determine has_anomaly_in_window: any value >= global_p95 or <= global_p5
    cond_high = df_sel['value'] >= df_sel['global_p95_value']
    cond_low  = df_sel['value'] <= df_sel['global_p5_value']
    df_flags = (
        df_sel.assign(is_anom = cond_high | cond_low)
        .groupby(['cmdb_id','trace_name'])['is_anom']
        .any()
        .reset_index()
        .rename(columns={'is_anom':'has_anomaly_in_window'})
    )

    agg = agg.merge(df_flags, on=['cmdb_id','trace_name'], how='left')
    agg['has_anomaly_in_window'] = agg['has_anomaly_in_window'].fillna(False)

    # format earliest timestamp to UTC+8 readable string
    agg['earliest_timestamp_in_window'] = agg['earliest_timestamp_in_window'].apply(
        lambda ts: pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    )

    # select and order columns
    agg_display = agg[[
        'cmdb_id','trace_name','sample_count_in_window','earliest_timestamp_in_window',
        'mean_value_in_window','max_value_in_window','global_p95_value','global_p5_value','has_anomaly_in_window'
    ]]

    # 5) Sort by sample_count desc, then max_value desc and take top 20
    agg_top = agg_display.sort_values(['sample_count_in_window','max_value_in_window'], ascending=[False, False]).head(20).reset_index(drop=True)

    # 6) summary counts
    distinct_groups = int(agg.shape[0])
    total_matched_rows = int(df_sel.shape[0])
else:
    agg_top = pd.DataFrame(columns=[
        'cmdb_id','trace_name','sample_count_in_window','earliest_timestamp_in_window',
        'mean_value_in_window','max_value_in_window','global_p95_value','global_p5_value','has_anomaly_in_window'
    ])
    distinct_groups = 0
    total_matched_rows = 0

summary_line = f"Matched groups={distinct_groups}, matched rows={total_matched_rows}"

# Return compact table and summary line
agg_top, summary_line

```
Out[5]:
```
Summary of trace analysis (window 2020-05-26 03:00:00 — 2020-05-26 03:30:00 UTC+8), filtering trace_name by substrings 'jdbc', 'remoteprocess', 'to_db', 'db':

- Totals:
  - Distinct (cmdb_id, trace_name) groups matched: 108
  - Total filtered rows matching substrings in window: 3,240

- Key observations (top aggregated rows, sorted by sample_count then max_value):
  1) docker_005 / trace.self.jdbc.row_count — samples=30, earliest=2020-05-26 03:00:00, mean≈1724.03, max=2160.0, global_p95=2135.50, global_p5=308.00, has_anomaly=True
  2) docker_001 / trace.self.jdbc.row_count — samples=30, earliest=2020-05-26 03:00:00, mean≈1379.93, max=1825.0, global_p95=1794.15, global_p5=254.40, has_anomaly=True
  3) docker_003 / trace.self.jdbc.row_count — samples=30, earliest=2020-05-26 03:00:00, mean≈1371.67, max=1816.0, global_p95=1773.10, global_p5=238.40, has_anomaly=True
  4) docker_002 / trace.self.jdbc.row_count — samples=30, earliest=2020-05-26 03:00:00, mean≈1380.23, max=1810.0, global_p95=1782.00, global_p5=174.40, has_anomaly=True
  5) docker_004 / trace.self.jdbc.row_count — samples=30, earliest=2020-05-26 03:00:00, mean≈1390.17, max=1777.0, global_p95=1748.10, global_p5=247.20, has_anomaly=True

- Additional notes:
  - Most top groups have 30 samples in the window (03:00–03:29) and all shown groups have has_anomaly_in_window = True (i.e., at least one value in-window reached/exceeded global P95 or was ≤ global P5).
  - Several strong relationships appear between docker nodes for remoteprocess row_count (e.g., to/from pairs), indicating correlated activity across pods.
  - Global P95/P5 were computed over each full series before filtering; aggregated statistics above show both global thresholds and windowed statistics.

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                     trace_name  sample_count_in_window earliest_timestamp_in_window  mean_value_in_window  max_value_in_window  global_p95_value  global_p5_value  has_anomaly_in_window
0   docker_005                      trace.self.jdbc.row_count                      30      2020-05-26 03:00:00 CST           1724.033333               2160.0           2135.50           308.00                   True
1   docker_001                      trace.self.jdbc.row_count                      30      2020-05-26 03:00:00 CST           1379.933333               1825.0           1794.15           254.40                   True
2   docker_003                      trace.self.jdbc.row_count                      30      2020-05-26 03:00:00 CST           1371.666667               1816.0           1773.10           238.40                   True
3   docker_002                      trace.self.jdbc.row_count                      30      2020-05-26 03:00:00 CST           1380.233333               1810.0           1782.00           174.40                   True
4   docker_004                      trace.self.jdbc.row_count                      30      2020-05-26 03:00:00 CST           1390.166667               1777.0           1748.10           247.20                   True
5   docker_007                      trace.self.jdbc.row_count                      30      2020-05-26 03:00:00 CST            850.833333               1137.0           1085.05           138.70                   True
6   docker_008                      trace.self.jdbc.row_count                      30      2020-05-26 03:00:00 CST            872.833333               1117.0           1068.00           128.70                   True
7   docker_003    trace.to_docker_005.remoteprocess.row_count                      30      2020-05-26 03:00:00 CST            342.400000                452.0            443.00            59.60                   True
8   docker_005  trace.from_docker_003.remoteprocess.row_count                      30      2020-05-26 03:00:00 CST            342.400000                452.0            443.00            59.60                   True
9   docker_004    trace.to_docker_005.remoteprocess.row_count                      30      2020-05-26 03:00:00 CST            347.200000                444.0            436.80            61.80                   True
10  docker_005  trace.from_docker_004.remoteprocess.row_count                      30      2020-05-26 03:00:00 CST            347.200000                444.0            436.80            61.80                   True
11  docker_001    trace.to_docker_007.remoteprocess.row_count                      30      2020-05-26 03:00:00 CST            169.633333                246.0            223.10            30.35                   True
12  docker_007  trace.from_docker_001.remoteprocess.row_count                      30      2020-05-26 03:00:00 CST            169.633333                246.0            223.10            30.35                   True
13  docker_001    trace.to_docker_008.remoteprocess.row_count                      30      2020-05-26 03:00:00 CST            175.166667                241.0            236.75            30.90                   True
14  docker_008  trace.from_docker_001.remoteprocess.row_count                      30      2020-05-26 03:00:00 CST            175.166667                241.0            236.75            30.90                   True
15  docker_002    trace.to_docker_007.remoteprocess.row_count                      30      2020-05-26 03:00:00 CST            170.766667                237.0            231.65            23.15                   True
16  docker_007  trace.from_docker_002.remoteprocess.row_count                      30      2020-05-26 03:00:00 CST            170.766667                237.0            231.65            23.15                   True
17  docker_002    trace.to_docker_008.remoteprocess.row_count                      30      2020-05-26 03:00:00 CST            173.900000                236.0            223.85            20.45                   True
18  docker_008  trace.from_docker_002.remoteprocess.row_count                      30      2020-05-26 03:00:00 CST            173.900000                236.0            223.85            20.45                   True
19  docker_001      trace.from_os_022.remoteprocess.row_count                      30      2020-05-26 03:00:00 CST             86.200000                114.0            112.10            15.90                   True, 'Matched groups=108, matched rows=3240')```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

# reuse existing df_metric and tz if present
if 'tz' not in globals():
    tz = pytz.timezone('Asia/Shanghai')
if 'df_metric' not in globals():
    df_metric = pd.read_csv('metric.csv')

# ensure types
df_metric['timestamp'] = df_metric['timestamp'].astype(int)
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# 1) Compute global P95 and P5 per (cmdb_id, kpi_name) over full series BEFORE filtering
thresholds_db = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(
        global_p95_value = lambda x: float(np.percentile(x.dropna(), 95)) if x.dropna().size>0 else np.nan,
        global_p5_value  = lambda x: float(np.percentile(x.dropna(), 5))  if x.dropna().size>0 else np.nan,
    )
    .reset_index()
)

# 2) Filter rows where cmdb_id begins with 'db_' and timestamp in incident window (UTC+8)
start_local = pd.Timestamp('2020-05-26 03:00:00').tz_localize(tz)
end_local   = pd.Timestamp('2020-05-26 03:30:00').tz_localize(tz)
start_ts = int(start_local.tz_convert('UTC').timestamp())
end_ts   = int(end_local.tz_convert('UTC').timestamp())

mask_db = df_metric['cmdb_id'].str.startswith('db_', na=False)
df_db_window = df_metric[mask_db & (df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)].copy()

# summary counts requested later
total_rows_db = int(df_db_window.shape[0])

# 2 continued: merge thresholds into windowed rows
df_db_w = df_db_window.merge(thresholds_db, on=['cmdb_id','kpi_name'], how='left')

# 2) Aggregate per (cmdb_id, kpi_name)
if not df_db_w.empty:
    agg = (
        df_db_w
        .groupby(['cmdb_id','kpi_name'])
        .agg(
            sample_count_in_window=('value','size'),
            earliest_timestamp_in_window=('timestamp','min'),
            mean_value_in_window=('value','mean'),
            max_value_in_window=('value','max'),
            min_value_in_window=('value','min'),
            global_p95_value=('global_p95_value','first'),
            global_p5_value =('global_p5_value','first'),
            # anomaly count in window: >= p95 or <= p5
        )
        .reset_index()
    )

    # compute anomaly_count_in_window using df_db_w flags
    df_db_w['is_high'] = df_db_w['value'] >= df_db_w['global_p95_value']
    df_db_w['is_low']  = df_db_w['value'] <= df_db_w['global_p5_value']
    df_flags = (
        df_db_w
        .assign(is_anom = df_db_w['is_high'] | df_db_w['is_low'])
        .groupby(['cmdb_id','kpi_name'])['is_anom']
        .agg(anomaly_count_in_window='sum')
        .reset_index()
    )
    agg = agg.merge(df_flags, on=['cmdb_id','kpi_name'], how='left')
    agg['anomaly_count_in_window'] = agg['anomaly_count_in_window'].fillna(0).astype(int)

    # 2) severity_score per group:
    def compute_severity(row):
        sev_high = np.nan
        sev_low  = np.nan
        # high severity: max_value / global_p95_value
        p95 = row['global_p95_value']
        if pd.notna(p95) and p95 != 0:
            sev_high = float(row['max_value_in_window']) / float(p95)
        # low severity: global_p5_value / min_value
        p5 = row['global_p5_value']
        minv = row['min_value_in_window']
        if pd.notna(minv) and minv != 0 and pd.notna(p5):
            sev_low = float(p5) / float(minv)
        # choose larger when both present, otherwise whichever is present
        vals = [v for v in (sev_high, sev_low) if pd.notna(v)]
        return float(max(vals)) if vals else np.nan

    agg['severity_score'] = agg.apply(compute_severity, axis=1)

    # format earliest timestamp to UTC+8 string
    agg['earliest_timestamp_in_window'] = agg['earliest_timestamp_in_window'].apply(
        lambda ts: pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    )

    # select and order columns
    agg_display = agg[[
        'cmdb_id','kpi_name','sample_count_in_window','earliest_timestamp_in_window',
        'mean_value_in_window','max_value_in_window','min_value_in_window',
        'global_p95_value','global_p5_value','anomaly_count_in_window','severity_score'
    ]]

    # 3) Sort by anomaly_count desc, then severity_score desc, return top 50
    agg_top = agg_display.sort_values(['anomaly_count_in_window','severity_score'], ascending=[False, False]).head(50).reset_index(drop=True)

    # 4) summary numbers
    total_groups_db = int(agg.shape[0])
else:
    agg_top = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','sample_count_in_window','earliest_timestamp_in_window',
        'mean_value_in_window','max_value_in_window','min_value_in_window',
        'global_p95_value','global_p5_value','anomaly_count_in_window','severity_score'
    ])
    total_groups_db = 0
    total_rows_db = 0

# Final compact outputs: aggregated top rows, and two summary numbers
agg_top, total_groups_db, total_rows_db

```
Out[6]:
```
summary = (
    "Summary of db_* KPI analysis (window 2020-05-26 03:00:00 — 2020-05-26 03:30:00 UTC+8):\n\n"
    f"- Total distinct (cmdb_id,kpi_name) groups in window: {total_groups_db}\n"
    f"- Total rows matched in window: {total_rows_db}\n\n"
    "Top example groups (by anomaly_count then severity):\n"
)

# take up to 5 top examples from agg_top if available
try:
    top_examples = agg_top.head(5)
    for i, row in top_examples.iterrows():
        summary += (
            f"{i+1}) {row['cmdb_id']} / {row['kpi_name']} — "
            f"samples={int(row['sample_count_in_window'])}, "
            f"anomalies={int(row['anomaly_count_in_window'])}, "
            f"max={row['max_value_in_window']:.6g}, "
            f"min={row['min_value_in_window']:.6g}, "
            f"global_p95={row['global_p95_value']:.6g}, "
            f"global_p5={row['global_p5_value']:.6g}, "
            f"severity={row['severity_score'] if pd.notna(row['severity_score']) else 'NaN'}, "
            f"earliest={row['earliest_timestamp_in_window']}\n"
        )
except Exception:
    summary += "No top examples available.\n"

summary += (
    "\nNotes:\n"
    "- Global P95/P5 were computed over the full series before filtering.\n"
    "- Many db_* groups show anomaly_count = 30 (all window samples flagged), indicating widespread deviation vs global thresholds.\n"
    "- severity_score may be very large when global_p95 is small; if global_p95 or min_value is zero, severity is reported as NaN.\n"
)

summary, agg_top.shape, total_groups_db, total_rows_db

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id             kpi_name  sample_count_in_window earliest_timestamp_in_window  mean_value_in_window  max_value_in_window  min_value_in_window  global_p95_value  global_p5_value  anomaly_count_in_window  severity_score
0   db_011          User_Commit                      30      2020-05-26 03:00:00 CST              0.016667              0.40000              0.00000           0.00250          0.00000                       30           160.0
1   db_005                 AIOS                      30      2020-05-26 03:00:00 CST              0.000667              0.01000              0.00000           0.00050          0.00000                       30            20.0
2   db_006  tnsping_result_time                      30      2020-05-26 03:00:00 CST              0.666667             10.00000              0.00000           0.50000          0.00000                       30            20.0
3   db_007  tnsping_result_time                      30      2020-05-26 03:00:00 CST              2.000000             20.00000              0.00000          10.00000          0.00000                       30             2.0
4   db_009                  AWS                      30      2020-05-26 03:00:00 CST              0.012000              0.04000              0.00000           0.02000          0.00000                       30             2.0
..     ...                  ...                     ...                          ...                   ...                  ...                  ...               ...              ...                      ...             ...
45  db_005  tnsping_result_time                      30      2020-05-26 03:00:00 CST              3.000000             10.00000              0.00000          10.00000          0.00000                       30             1.0
46  db_006                 Hang                      30      2020-05-26 03:00:00 CST              1.000000              1.00000              1.00000           1.00000          1.00000                       30             1.0
47  db_006        Login_Per_Sec                      30      2020-05-26 03:00:00 CST              0.110000              0.20000              0.00000           0.20000          0.10000                       30             1.0
48  db_006            MEM_Total                      30      2020-05-26 03:00:00 CST            508.252410            508.25241            508.25241         508.25241        508.25241                       30             1.0
49  db_006         On_Off_State                      30      2020-05-26 03:00:00 CST              1.000000              1.00000              1.00000           1.00000          1.00000                       30             1.0

[50 rows x 11 columns], 587, 13632)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

# reuse timezone and df_trace if available
if 'tz' not in globals():
    tz = pytz.timezone('Asia/Shanghai')
if 'df_trace' not in globals():
    df_trace = pd.read_csv('trace.csv')

# ensure types
df_trace['timestamp'] = df_trace['timestamp'].astype(int)
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')

# ensure incident window epoch seconds (UTC+8)
if 'start_ts' not in globals() or 'end_ts' not in globals():
    start_local = pd.Timestamp('2020-05-26 03:00:00').tz_localize(tz)
    end_local   = pd.Timestamp('2020-05-26 03:30:00').tz_localize(tz)
    start_ts = int(start_local.tz_convert('UTC').timestamp())
    end_ts   = int(end_local.tz_convert('UTC').timestamp())

# 1) filter to incident window
df_window = df_trace[(df_trace['timestamp'] >= start_ts) & (df_trace['timestamp'] <= end_ts)].copy()

# 2) select trace_name values that contain 'db_' or 'to_db' (case-sensitive)
mask = df_window['trace_name'].str.contains('db_|to_db', na=False)
df_sel = df_window[mask].copy()

# 3) aggregate per (cmdb_id, trace_name)
if not df_sel.empty:
    agg = (
        df_sel
        .groupby(['cmdb_id','trace_name'])
        .agg(
            sample_count_in_window=('value','size'),
            earliest_timestamp_in_window=('timestamp','min'),
            max_value_in_window=('value','max'),
            mean_value_in_window=('value','mean'),
        )
        .reset_index()
    )

    # format earliest timestamp to UTC+8 readable string
    agg['earliest_timestamp_in_window'] = agg['earliest_timestamp_in_window'].apply(
        lambda ts: pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    )

    # reorder and round numeric columns for compactness
    agg_display = agg[[
        'cmdb_id','trace_name','sample_count_in_window','earliest_timestamp_in_window',
        'max_value_in_window','mean_value_in_window'
    ]]
    agg_display['max_value_in_window'] = agg_display['max_value_in_window'].astype(float).round(6)
    agg_display['mean_value_in_window'] = agg_display['mean_value_in_window'].astype(float).round(6)

    # 4) sort by sample_count desc then max_value desc, limit to top 50
    agg_top = agg_display.sort_values(['sample_count_in_window','max_value_in_window'], ascending=[False, False]).head(50).reset_index(drop=True)

    # summary numbers
    distinct_groups = int(agg.shape[0])
    total_matched_rows = int(df_sel.shape[0])
else:
    agg_top = pd.DataFrame(columns=[
        'cmdb_id','trace_name','sample_count_in_window','earliest_timestamp_in_window',
        'max_value_in_window','mean_value_in_window'
    ])
    distinct_groups = 0
    total_matched_rows = 0

summary_line = f"Matched groups={distinct_groups}, matched rows={total_matched_rows}"

# display compact results
agg_top, summary_line

```
Out[7]:
```
summary = (
    "Result summary:\n"
    "- Incident window: 2020-05-26 03:00:00 to 2020-05-26 03:30:00 (UTC+8).\n"
    "- After filtering trace.csv rows in that window, no trace_name values contained the substrings 'db_' or 'to_db' (case-sensitive).\n"
    "- Matched groups = 0, matched rows = 0.\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [cmdb_id, trace_name, sample_count_in_window, earliest_timestamp_in_window, max_value_in_window, mean_value_in_window]
Index: [], 'Matched groups=0, matched rows=0')```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz

# reuse existing variables if present
if 'tz' not in globals():
    tz = pytz.timezone('Asia/Shanghai')
if 'df_metric' not in globals():
    df_metric = pd.read_csv('metric.csv')

# ensure correct dtypes
df_metric['timestamp'] = df_metric['timestamp'].astype(int)
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# ensure incident window epoch seconds (UTC+8). reuse if available
if 'start_ts' not in globals() or 'end_ts' not in globals():
    start_local = pd.Timestamp('2020-05-26 03:00:00').tz_localize(tz)
    end_local   = pd.Timestamp('2020-05-26 03:30:00').tz_localize(tz)
    start_ts = int(start_local.tz_convert('UTC').timestamp())
    end_ts   = int(end_local.tz_convert('UTC').timestamp())

# 1) Compute global P95 and P5 per (cmdb_id, kpi_name) over the full series (reuse thresholds_db if exists)
if 'thresholds_db' in globals():
    thresholds = thresholds_db.copy()
else:
    thresholds = (
        df_metric
        .groupby(['cmdb_id','kpi_name'])['value']
        .agg(
            global_p95_value = lambda x: float(np.percentile(x.dropna(), 95)) if x.dropna().size>0 else np.nan,
            global_p5_value  = lambda x: float(np.percentile(x.dropna(), 5))  if x.dropna().size>0 else np.nan,
        )
        .reset_index()
    )
    thresholds_db = thresholds  # cache for reuse

# 2) Filter db_* rows in incident window
mask_db = df_metric['cmdb_id'].str.startswith('db_', na=False)
df_db_window = df_metric[mask_db & (df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)].copy()

# count distinct db components in window (for summary)
total_db_components_in_window = int(df_db_window['cmdb_id'].nunique())

# merge thresholds into window rows
if not df_db_window.empty:
    df_db_w = df_db_window.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')

    # mark anomaly points in-window
    df_db_w['is_high'] = df_db_w['value'] >= df_db_w['global_p95_value']
    df_db_w['is_low']  = df_db_w['value'] <= df_db_w['global_p5_value']
    df_db_w['is_anom'] = df_db_w['is_high'] | df_db_w['is_low']

    # 2) Per-(cmdb_id,kpi_name) KPI-level aggregation using anomaly rows
    # compute sample_count_in_window also (total samples in window for KPI)
    kpi_agg_all = (
        df_db_w
        .groupby(['cmdb_id','kpi_name'])
        .agg(
            sample_count_in_window=('value','size'),
            anomaly_count_in_window=('is_anom','sum'),
        )
        .reset_index()
    )

    # aggregate anomaly-specific stats (only from anomaly rows)
    df_k_anom = df_db_w[df_db_w['is_anom']].copy()
    if not df_k_anom.empty:
        kpi_anom_stats = (
            df_k_anom
            .groupby(['cmdb_id','kpi_name'])
            .agg(
                earliest_anomaly_timestamp=('timestamp','min'),
                max_anom_value=('value','max'),
                min_anom_value=('value','min'),
            )
            .reset_index()
        )
    else:
        # empty frame with expected columns
        kpi_anom_stats = pd.DataFrame(columns=['cmdb_id','kpi_name','earliest_anomaly_timestamp','max_anom_value','min_anom_value'])

    # join with global thresholds
    kpi_agg = kpi_agg_all.merge(kpi_anom_stats, on=['cmdb_id','kpi_name'], how='left')
    kpi_agg = kpi_agg.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')

    # compute severity_score per KPI based on anomaly rows
    def kpi_severity(row):
        if row['anomaly_count_in_window'] <= 0:
            return np.nan
        sev_high = np.nan
        sev_low = np.nan
        p95 = row['global_p95_value']
        p5 = row['global_p5_value']
        maxv = row.get('max_anom_value', np.nan)
        minv = row.get('min_anom_value', np.nan)
        # high severity
        if pd.notna(maxv) and pd.notna(p95) and p95 != 0:
            sev_high = float(maxv) / float(p95)
        # low severity
        if pd.notna(minv) and pd.notna(p5) and minv != 0:
            sev_low = float(p5) / float(minv)
        vals = [v for v in (sev_high, sev_low) if pd.notna(v)]
        return float(max(vals)) if vals else np.nan

    kpi_agg['severity_score'] = kpi_agg.apply(kpi_severity, axis=1)

    # convert earliest_anomaly_timestamp to readable UTC+8 string
    kpi_agg['earliest_anomaly_timestamp'] = kpi_agg['earliest_anomaly_timestamp'].apply(
        lambda ts: pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
        if pd.notna(ts) else pd.NaT
    )

    # 2) Aggregate across KPIs per db
    def choose_kpi_with_max_sev(df):
        # pick KPI with largest severity_score; if tie, pick first by severity then anomaly_count
        df_valid = df[df['anomaly_count_in_window']>0].copy()
        if df_valid.empty:
            return pd.Series({
                'distinct_anomalous_kpis': 0,
                'total_anomaly_points': 0,
                'earliest_anomaly_timestamp_across_kpis': pd.NaT,
                'kpi_with_max_severity': None,
                'max_severity_score': np.nan
            })
        distinct_anomalous_kpis = int((df_valid['anomaly_count_in_window']>0).sum())
        total_anomaly_points = int(df_valid['anomaly_count_in_window'].sum())
        # earliest timestamp across anomalous KPIs (skip NaT)
        times = df_valid['earliest_anomaly_timestamp'].dropna()
        if not times.empty:
            earliest_ts = times.min()
        else:
            earliest_ts = pd.NaT
        # pick kpi with max severity (NaNs last)
        df_valid['severity_filled'] = df_valid['severity_score'].fillna(-np.inf)
        row_max = df_valid.sort_values(['severity_filled','anomaly_count_in_window'], ascending=[False,False]).iloc[0]
        kpi_name = row_max['kpi_name']
        max_sev = row_max['severity_score']
        return pd.Series({
            'distinct_anomalous_kpis': distinct_anomalous_kpis,
            'total_anomaly_points': total_anomaly_points,
            'earliest_anomaly_timestamp_across_kpis': earliest_ts,
            'kpi_with_max_severity': kpi_name,
            'max_severity_score': max_sev
        })

    db_agg = (
        kpi_agg
        .groupby('cmdb_id')
        .apply(choose_kpi_with_max_sev)
        .reset_index()
    )

    # format earliest timestamp string
    def fmt_ts(x):
        if pd.isna(x):
            return pd.NaT
        try:
            return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
        except Exception:
            return pd.NaT

    db_agg['earliest_anomaly_timestamp_across_kpis'] = db_agg['earliest_anomaly_timestamp_across_kpis'].apply(fmt_ts)

    # sort and keep top 20
    db_agg_top = db_agg.sort_values(['total_anomaly_points','max_severity_score'], ascending=[False, False]).head(20).reset_index(drop=True)

    # total anomaly points across all db_*
    total_anomaly_points_all = int(kpi_agg['anomaly_count_in_window'].sum())
else:
    # no db rows in window
    db_agg_top = pd.DataFrame(columns=[
        'cmdb_id','distinct_anomalous_kpis','total_anomaly_points','earliest_anomaly_timestamp_across_kpis',
        'kpi_with_max_severity','max_severity_score'
    ])
    total_db_components_in_window = 0
    total_anomaly_points_all = 0

# Final compact outputs: db_agg_top (table), total_db_components_in_window, total_anomaly_points_all
db_agg_top, total_db_components_in_window, total_anomaly_points_all

```
Out[8]:
```
Summary of db_* aggregation within 2020-05-26 03:00:00–03:30:00 (UTC+8):

Totals
- Distinct db_* components found in the window: 13
- Total anomaly points across all db_*: 7,665

Top per-db results (sorted by total_anomaly_points desc; showing available rows):
1) db_006 — distinct_anomalous_kpis=45, total_anomaly_points=733, kpi_with_max_severity=tnsping_result_time, max_severity_score=20.0
2) db_013 — distinct_anomalous_kpis=45, total_anomaly_points=668, kpi_with_max_severity=Redo_Per_Sec, max_severity_score≈41.72
3) db_011 — distinct_anomalous_kpis=45, total_anomaly_points=634, kpi_with_max_severity=User_Commit, max_severity_score=160.0
4) db_005 — distinct_anomalous_kpis=45, total_anomaly_points=616, kpi_with_max_severity=AIOS, max_severity_score=20.0
5) db_010 — distinct_anomalous_kpis=45, total_anomaly_points=614, kpi_with_max_severity=Redo_Per_Sec, max_severity_score≈105.40
6) db_001 — distinct_anomalous_kpis=45, total_anomaly_points=590, kpi_with_max_severity=Logic_Read_Per_Sec, max_severity_score≈34.79
7) db_004 — distinct_anomalous_kpis=45, total_anomaly_points=579, kpi_with_max_severity=ACS, max_severity_score≈1.46
8) db_007 — distinct_anomalous_kpis=45, total_anomaly_points=574, kpi_with_max_severity=Redo_Per_Sec, max_severity_score≈64.17
9) db_009 — distinct_anomalous_kpis=44, total_anomaly_points=573, kpi_with_max_severity=AWS, max_severity_score=2.0
10) db_008 — distinct_anomalous_kpis=45, total_anomaly_points=571, kpi_with_max_severity=Sess_Active, max_severity_score=1.5
11) db_002 — distinct_anomalous_kpis=45, total_anomaly_points=536, kpi_with_max_severity=Physical_Read_Per_Sec, max_severity_score≈55.33
12) db_012 — distinct_anomalous_kpis=45, total_anomaly_points=521, kpi_with_max_severity=Redo_Per_Sec, max_severity_score≈2.96
13) db_003 — distinct_anomalous_kpis=46, total_anomaly_points=456, kpi_with_max_severity=Redo_Per_Sec, max_severity_score≈2.14

Notes
- Global P95/P5 were computed over each full (cmdb_id,kpi_name) series before any filtering; in-window anomaly points were marked where value >= global_P95 or value <= global_P5.
- earliest_anomaly_timestamp_across_kpis is shown as NaT in the returned table (no consolidated earliest timestamp was produced in the aggregated output).
- severity_score is the largest KPI-level severity (high or low) per DB; NaN may appear where division-by-zero occurred for a KPI's threshold.

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id  distinct_anomalous_kpis  total_anomaly_points earliest_anomaly_timestamp_across_kpis  kpi_with_max_severity  max_severity_score
0   db_006                       45                   733                                    NaT    tnsping_result_time           20.000000
1   db_013                       45                   668                                    NaT           Redo_Per_Sec           41.719067
2   db_011                       45                   634                                    NaT            User_Commit          160.000000
3   db_005                       45                   616                                    NaT                   AIOS           20.000000
4   db_010                       45                   614                                    NaT           Redo_Per_Sec          105.399055
5   db_001                       45                   590                                    NaT     Logic_Read_Per_Sec           34.786216
6   db_004                       45                   579                                    NaT                    ACS            1.463415
7   db_007                       45                   574                                    NaT           Redo_Per_Sec           64.165441
8   db_009                       44                   573                                    NaT                    AWS            2.000000
9   db_008                       45                   571                                    NaT            Sess_Active            1.500000
10  db_002                       45                   536                                    NaT  Physical_Read_Per_Sec           55.333333
11  db_012                       45                   521                                    NaT           Redo_Per_Sec            2.964023
12  db_003                       46                   456                                    NaT           Redo_Per_Sec            2.144358, 13, 7665)```
```